In [1]:
!pip install -U "typing_extensions>=4.12" vllm transformers datasets accelerate sentencepiece

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
import json
import random
import ast
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

# ----------------------------
# Config
# ----------------------------
USE_SAMPLE = False
SAMPLE_SIZE = 500
SEED = 42

SELECTED_MODEL_ALIASES = [
    "qwen2.5-0.5b",
    "qwen2.5-1.5b",
    "qwen2.5-3b",
    "smollm2-1.7b",
    "granite-3.3-2b",
    "nemotron-mini-4b",
    "phi-3.5-mini",
]

MODEL_SPECS = [
    {"alias": "qwen2.5-0.5b", "hf_id": "Qwen/Qwen2.5-0.5B-Instruct", "trust_remote_code": False},
    {"alias": "qwen2.5-1.5b", "hf_id": "Qwen/Qwen2.5-1.5B-Instruct", "trust_remote_code": False},
    {"alias": "qwen2.5-3b", "hf_id": "Qwen/Qwen2.5-3B-Instruct", "trust_remote_code": False},
    {"alias": "smollm2-1.7b", "hf_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "trust_remote_code": False},
    {"alias": "granite-3.3-2b", "hf_id": "ibm-granite/granite-3.3-2b-instruct", "trust_remote_code": False},
    {"alias": "nemotron-mini-4b", "hf_id": "nvidia/Nemotron-Mini-4B-Instruct", "trust_remote_code": False},
    {"alias": "phi-3.5-mini", "hf_id": "microsoft/Phi-3.5-mini-instruct", "trust_remote_code": True},
]

selected_model_specs = [
    spec for spec in MODEL_SPECS
    if spec["alias"] in SELECTED_MODEL_ALIASES
]

# ----------------------------
# Prompt pieces
# ----------------------------
FEW_SHOT_1_USER = """Subject: Group Messaging for Admissions Process
Good morning, everyone,
I hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:

wynqvrh053 - Meeting at 10:20am
luka.burg - Meeting at 21
qahl.wittauer - Meeting at quarter past 13
gholamhossein.ruschke - Meeting at 9:47 PM
pdmjrsyoz1460"""

FEW_SHOT_1_ASSISTANT = """{
  "items": [
    { "value": "wynqvrh053", "label": "USERNAME" },
    { "value": "10:20am", "label": "TIME" },
    { "value": "luka.burg", "label": "USERNAME" },
    { "value": "21", "label": "TIME" },
    { "value": "qahl.wittauer", "label": "USERNAME" },
    { "value": "quarter past 13", "label": "TIME" },
    { "value": "gholamhossein.ruschke", "label": "USERNAME" },
    { "value": "9:47 PM", "label": "TIME" },
    { "value": "pdmjrsyoz1460", "label": "USERNAME" }
  ]
}"""

FEW_SHOT_2_USER = 'Card: KB90324ER\n Country: GB\n Building: 163\n Street: Conygre Grove\n City: Bristol\n State: ENG\n Postcode: BS34 7HU, BS34 7HZ\n Password: q4R\\n\n2. Applicant: Baasgaran Palmoso\n Email: blerenbaasgara@gmail.com\n Social Number: 107-393-9036\n ID Card: SC78428CU\n Country: United Kingdom\n Building: 646\n Street: School Lane\n City: Altrincham\n State: ENG\n Postcode: WA14 5R'

FEW_SHOT_2_ASSISTANT = """{
  "items": [
    { "value": "KB90324ER", "label": "ID" },
    { "value": "GB", "label": "COUNTRY" },
    { "value": "163", "label": "BUILDING" },
    { "value": "Conygre Grove", "label": "STREET" },
    { "value": "Bristol", "label": "CITY" },
    { "value": "ENG", "label": "STATE" },
    { "value": "BS34 7HU, BS34 7HZ", "label": "POSTCODE" },
    { "value": "q4R\\\\", "label": "PASS" },
    { "value": "Baasgaran", "label": "LASTNAME" },
    { "value": "Palmoso", "label": "LASTNAME" },
    { "value": "blerenbaasgara@gmail.com", "label": "EMAIL" },
    { "value": "107-393-9036", "label": "SOCIALNUMBER" },
    { "value": "SC78428CU", "label": "ID" },
    { "value": "United Kingdom", "label": "COUNTRY" },
    { "value": "646", "label": "BUILDING" },
    { "value": "School Lane", "label": "STREET" },
    { "value": "Altrincham", "label": "CITY" },
    { "value": "ENG", "label": "STATE" }
  ]
}"""

labels_list = [
    "EMAIL", "PHONE", "IP", "IPV6", "MAC", "CARD", "IBAN",
    "NAME", "FIRSTNAME", "LASTNAME", "LOCATION", "ORG", "DATE",
    "USERNAME", "TIME", "ID", "COUNTRY", "BUILDING", "STREET",
    "CITY", "STATE", "POSTCODE", "PASS", "SOCIALNUMBER",
]
labels_str = ", ".join(labels_list)

system_prompt = f"""You are a PII scrubbing assistant operating as the final stage of a detection pipeline.

CONTEXT — earlier pipeline stages have already redacted some PII by replacing it with bracketed placeholders. The placeholders you may encounter are:
[FIRSTNAME] [LASTNAME] [NAME] [EMAIL] [PHONE] [IP] [IPV6] [MAC] [CARD] [IBAN] [LOCATION] [ORG] [DATE] [TIME] [USERNAME] [ID] [COUNTRY] [BUILDING] [STREET] [CITY] [STATE] [POSTCODE] [PASS] [SOCIALNUMBER]

Do NOT return any of these placeholders as matches — they are already redacted.

BACKGROUND — to help you recognise what counts as PII, here is a broad taxonomy of PII categories that earlier detectors cover. Anything in this taxonomy that is still present as real text in the input is your target:
- Personal: first/last name, date of birth, age, gender, sexuality, race/ethnicity, religion, political view, occupation, education
- Contact: email, phone, street address, city, county, state, country, coordinates, zip code, PO box
- Financial: credit/debit card, CVV, bank routing number, account number, IBAN, SWIFT/BIC, PIN, SSN, tax ID, EIN
- Government IDs: passport, driver's licence, licence plate, national ID, voter ID
- Digital: IPv4, IPv6, MAC address, URL, username, password, device ID, IMEI, serial number, API key, secret key
- Healthcare: medical record number, health plan ID, blood type, biometric identifier, health condition, medication, insurance policy
- Temporal: date, time, datetime
- Organisation: company name, employee ID, customer ID, certificate/licence number, vehicle identifier

YOUR TASK — find any remaining unredacted PII of these output types only: {labels_str}.

Return a JSON object with an "items" array. Each item must have:
- "value": the exact substring from the text (character-for-character, copy it exactly)
- "label": one of {labels_str}

Rules:
- Only mark PII that is explicitly present as real text, not inside a [PLACEHOLDER].
- Never reconstruct or infer content that is hidden behind a placeholder.
- If unsure whether something is PII, do not include it.
- Return all items, even if there are many. It is OK if items length is 200+.
- Return {{"items": []}} only if there is no remaining unredacted PII of these types."""

# ----------------------------
# Data
# ----------------------------
dataset = load_dataset("nvidia/Nemotron-PII")
all_texts = list(dataset["train"]["text"])
all_spans = list(dataset["train"]["spans"])

if USE_SAMPLE:
    random.seed(SEED)
    indices = random.sample(range(len(all_texts)), SAMPLE_SIZE)
    eval_texts = [all_texts[i] for i in indices]
    eval_spans = [all_spans[i] for i in indices]
else:
    eval_texts = all_texts
    eval_spans = all_spans

# ----------------------------
# Helpers
# ----------------------------
def build_messages(text: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": FEW_SHOT_1_USER},
        {"role": "assistant", "content": FEW_SHOT_1_ASSISTANT},
        {"role": "user", "content": FEW_SHOT_2_USER},
        {"role": "assistant", "content": FEW_SHOT_2_ASSISTANT},
        {"role": "user", "content": text},
    ]

def build_gt_json(span_str: str) -> str:
    # spans are stored as stringified Python objects in this dataset
    span_list = ast.literal_eval(span_str)
    items = []
    for s in span_list:
        text = s.get("text")
        label = s.get("label")
        if isinstance(text, str) and isinstance(label, str):
            items.append({"value": text, "label": label})
    return json.dumps({"items": items}, ensure_ascii=False)

def pct(arr, q):
    return int(np.percentile(arr, q)) if len(arr) else 0

def ceil_to_multiple(x: int, base: int = 64) -> int:
    return int(base * np.ceil(x / base))

def analyze_token_budget_for_model(model_spec, eval_texts, eval_spans):
    tokenizer = AutoTokenizer.from_pretrained(
        model_spec["hf_id"],
        trust_remote_code=model_spec["trust_remote_code"],
    )

    prompt_lens = []
    gt_output_lens = []
    total_lens = []

    for text, span in zip(eval_texts, eval_spans):
        messages = build_messages(text)
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompt_len = len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])

        gt_json = build_gt_json(span)
        gt_output_len = len(tokenizer(gt_json, return_tensors="pt")["input_ids"][0])

        prompt_lens.append(prompt_len)
        gt_output_lens.append(gt_output_len)
        total_lens.append(prompt_len + gt_output_len)

    prompt_lens = np.array(prompt_lens)
    gt_output_lens = np.array(gt_output_lens)
    total_lens = np.array(total_lens)

    max_prompt = int(prompt_lens.max())
    p95_prompt = pct(prompt_lens, 95)
    p99_prompt = pct(prompt_lens, 99)

    max_gt = int(gt_output_lens.max())
    p95_gt = pct(gt_output_lens, 95)
    p99_gt = pct(gt_output_lens, 99)

    max_total = int(total_lens.max())
    p95_total = pct(total_lens, 95)
    p99_total = pct(total_lens, 99)

    # Recommendations:
    # - "recommended": high coverage, practical
    # - "conservative": no truncation on this sampled set
    rec_max_new_tokens = ceil_to_multiple(p99_gt + 16, 32)
    cons_max_new_tokens = ceil_to_multiple(max_gt + 16, 32)

    rec_max_model_len = ceil_to_multiple(max_prompt + rec_max_new_tokens, 64)
    cons_max_model_len = ceil_to_multiple(max_prompt + cons_max_new_tokens, 64)

    return {
        "model": model_spec["alias"],
        "hf_id": model_spec["hf_id"],

        "prompt_tokens_max": max_prompt,
        "prompt_tokens_p95": p95_prompt,
        "prompt_tokens_p99": p99_prompt,

        "gt_output_tokens_max": max_gt,
        "gt_output_tokens_p95": p95_gt,
        "gt_output_tokens_p99": p99_gt,

        "total_tokens_max": max_total,
        "total_tokens_p95": p95_total,
        "total_tokens_p99": p99_total,

        "recommended_max_new_tokens": rec_max_new_tokens,
        "conservative_max_new_tokens": cons_max_new_tokens,

        "recommended_max_model_len": rec_max_model_len,
        "conservative_max_model_len": cons_max_model_len,
    }

# ----------------------------
# Run analysis
# ----------------------------
budget_results = []
for spec in selected_model_specs:
    print(f"Analyzing token budgets for {spec['alias']} ...")
    budget_results.append(analyze_token_budget_for_model(spec, eval_texts, eval_spans))

# Raw structured results
budget_results

Analyzing token budgets for qwen2.5-0.5b ...
Analyzing token budgets for qwen2.5-1.5b ...
Analyzing token budgets for qwen2.5-3b ...
Analyzing token budgets for smollm2-1.7b ...
Analyzing token budgets for granite-3.3-2b ...
Analyzing token budgets for nemotron-mini-4b ...
Analyzing token budgets for phi-3.5-mini ...


[{'model': 'qwen2.5-0.5b',
  'hf_id': 'Qwen/Qwen2.5-0.5B-Instruct',
  'prompt_tokens_max': 3354,
  'prompt_tokens_p95': 2058,
  'prompt_tokens_p99': 2271,
  'gt_output_tokens_max': 2968,
  'gt_output_tokens_p95': 316,
  'gt_output_tokens_p99': 475,
  'total_tokens_max': 5937,
  'total_tokens_p95': 2335,
  'total_tokens_p99': 2659,
  'recommended_max_new_tokens': 512,
  'conservative_max_new_tokens': 3008,
  'recommended_max_model_len': 3904,
  'conservative_max_model_len': 6400},
 {'model': 'qwen2.5-1.5b',
  'hf_id': 'Qwen/Qwen2.5-1.5B-Instruct',
  'prompt_tokens_max': 3354,
  'prompt_tokens_p95': 2058,
  'prompt_tokens_p99': 2271,
  'gt_output_tokens_max': 2968,
  'gt_output_tokens_p95': 316,
  'gt_output_tokens_p99': 475,
  'total_tokens_max': 5937,
  'total_tokens_p95': 2335,
  'total_tokens_p99': 2659,
  'recommended_max_new_tokens': 512,
  'conservative_max_new_tokens': 3008,
  'recommended_max_model_len': 3904,
  'conservative_max_model_len': 6400},
 {'model': 'qwen2.5-3b',
  'hf